# 01 — Gegnerstruktur

Frage: *Unterscheidet sich die Gegnerstruktur zwischen female_top (ELO 2400–2600) und male_control (age-matched)?*

Kennzahl: `avg_opponent_diff` = Ø Rating der Gegner − eigenes Rating pro Periode.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import load_view, load_query, apply_style, GROUP_PALETTE, GROUP_ORDER

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

apply_style()

## Daten laden

In [ ]:
df = load_view('v_opponent_strength')
df['period'] = pd.to_datetime(df['period'])
df[['avg_opponent_rating', 'avg_opponent_diff', 'own_rating']] = (
    df[['avg_opponent_rating', 'avg_opponent_diff', 'own_rating']].astype(float)
)
print(df.shape)
df.head()

## Summary pro Gruppe

In [ ]:
summary = df.groupby('analysis_group').agg(
    n_player_periods=('fide_id', 'count'),
    n_players=('fide_id', 'nunique'),
    mean_opp_rating=('avg_opponent_rating', 'mean'),
    mean_opp_diff=('avg_opponent_diff', 'mean'),
    median_opp_diff=('avg_opponent_diff', 'median'),
).round(1)
summary

## Boxplot — Ø Rating-Differenz pro Gruppe

Negative Werte = Gegner im Schnitt schwächer als man selbst.

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    data=df, x='analysis_group', y='avg_opponent_diff',
    order=GROUP_ORDER, palette=GROUP_PALETTE, ax=ax,
)
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_ylabel('Ø Gegnerrating − eigenes Rating')
ax.set_xlabel('')
ax.set_title('Gegnerstruktur pro (Spieler, Periode)')
plt.tight_layout(); plt.show()

## Verteilung der einzelnen Gegner-Ratings

Für einen direkten Histogramm-Vergleich gehen wir auf Einzelpartie-Ebene.

In [ ]:
raw = load_query('''
    SELECT p.analysis_group, gr.opponent_rating
    FROM game_results gr
    JOIN players p USING (fide_id)
    WHERE p.active = TRUE AND p.analysis_group IS NOT NULL
      AND gr.opponent_rating IS NOT NULL
''')
fig, ax = plt.subplots()
sns.histplot(
    data=raw, x='opponent_rating', hue='analysis_group',
    bins=40, stat='density', common_norm=False,
    palette=GROUP_PALETTE, hue_order=GROUP_ORDER, ax=ax,
)
ax.set_xlabel('Gegner-Rating')
ax.set_title('Verteilung der Gegner-Ratings')
plt.tight_layout(); plt.show()